# BP4 Gate 2 — Data Verification & Feature/Taxonomy Engineering
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Purpose
Builds the two real journey units BP4 Gate 1's `policy.json` already defined — never a third,
different design invented here. This notebook does the actual engineering: it reads Gate 1's real
policy live (never re-derives BP4's scope from scratch), applies an explicit, named null-sentinel
fill to every journey-relevant column with a real null (Gate 2's own exit criterion, "zero nulls
silently dropped"), builds and writes the real Gold-layer journey tables, and writes the
feature-lineage table Section 8 names as this gate's own required artifact.

## What this notebook does, concretely
1. **Reads Gate 1's real `policy.json` live** and cross-checks this run's own live measurements
   against it (row count, null counts, issue-cluster count) — never assumes Gate 1's numbers still
   hold without re-measuring.
2. **Builds the complaint-event journey** (row-level): parses `Date received` / `Date sent to
   company`, computes the real `response_lag_days` duration and a real `complaint_month` bucket,
   joins in the real Gold-layer `common_taxonomy_bucket` overlay (BP1/BP2's own Gate 2 work,
   reused unmodified — HYPER), and applies the explicit sentinel fill below.
3. **Builds the issue-cluster journey** (aggregate-level) as two real Gold tables: a monthly
   complaint-volume time series per real `(Company, Product, Sub-product, Issue, Sub-issue)`
   cluster, and a one-row-per-cluster summary (total count, real first/last active date, real
   active-month count, mean response lag, real BANKING77-overlay coverage fraction, a real
   `is_recurring_cluster` flag) — WARP: Polars lazy group-by push-down throughout, no per-row
   Python loop, matching BP4's own designated Section 17.5 tech stack.
4. **Writes the feature-lineage table** (`gate2_feature_lineage.csv`) — one row per engineered
   column, its real source column(s), its transform, and its null-handling rule — built
   programmatically from the same module constants the pipeline itself uses, so it can never drift
   from what the pipeline actually does.

## The explicit null-sentinel fill (Gate 2's own "zero nulls silently dropped" exit criterion)
Gate 1 live-found real nulls in three journey-relevant columns: `Sub-product` (21 rows),
`Sub-issue` (25,808 rows), `State` (2,193 rows). `Sub-product` and `Sub-issue` are part of BP4's
own issue-cluster key — left as real nulls, a null-unaware downstream tool (an Excel pivot, a
different aggregation engine than Polars) could silently drop or misgroup those rows differently
than Polars' own group-by treats them. This notebook closes that risk explicitly rather than
relying on Polars' implicit "nulls group together" behavior: every null in these three columns is
cast to a named sentinel category (`MISSING_SUB_PRODUCT`, `MISSING_SUB_ISSUE`, `MISSING_STATE` —
HYPER-reusing BP3's own `NULL_SENTINEL_MAP` naming convention verbatim for cross-BP consistency),
with a companion boolean `*_was_null` flag column recording which rows were filled, so a filled
row is always traceable and never silently indistinguishable from a genuine real category. Because
Polars already groups real nulls together under the hood, this fill is expected to produce the
*same* real cluster count Gate 1 already found (37,160) — verified live as a structural check
below, not assumed.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run
  it on your own machine, and the real, live-checked results below become this project's Gate 2
  record for BP4.
- **Zero-fabrication** (Section 12.1): every count, null total, and cluster statistic below is
  read live from the real raw CFPB file and the real Gold-layer taxonomy parquet — never asserted
  from Gate 1's policy.json without re-measuring.
- **CFPB supervisory & complaint-handling standards / GLBA (Master Plan Section 9)**: this real
  extract carries no narrative-text column at all — re-verified live below (Section 5), not
  assumed carried over from Gate 1 — so the Gate 2 compliance touchpoint ("PII screen on narrative
  text before any downstream/external call") has no narrative text to screen; stated honestly
  rather than silently skipped.
- **WARP**: `configure_performance()` first. Every group-by and join below is a lazy Polars
  operation, collected only where a live number or a Gold-layer write requires it — matching BP4's
  own designated Section 17.5 aggregation stack (Polars lazy scans, SQL-style group-by push-down,
  no per-row Python loop) from Gate 2 forward, not introduced later.
- **HYPER**: builds the new shared module `src/features/bp4_journey_features.py` AT this gate
  (matching BP3's own improved pattern — not retroactively extracted at Gate 6 the way BP2's
  was), reuses `src/utils/bp1_config_sync.py` (generic marker-based config read/write, already
  reused unmodified by BP1/BP2/BP3) and `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` unmodified, and
  reuses BP3's own `NULL_SENTINEL_MAP` sentinel-naming convention verbatim.
- **Idempotent**: re-running this notebook overwrites `configs/bp4_customer_journey_analytics.yaml`
  (Gate 2's own marker block only — Gate 1's front matter and every other gate's block are
  preserved verbatim regardless of position), this notebook's `gate2_feature_lineage.csv`, and the
  three Gold-layer Parquet outputs below, in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Outputs (idempotent overwrite-in-place)
- `src/features/bp4_journey_features.py` — new shared module (HYPER)
- `notebooks/bp4_customer_journey_analytics/artifacts/gate2_feature_lineage.csv`
- `data/processed/cfpb_journey_event_gold.parquet` — complaint-event journey, every real row
  tagged, none dropped
- `data/processed/cfpb_issue_cluster_monthly_gold.parquet` — real monthly complaint-volume time
  series per issue-cluster
- `data/processed/cfpb_issue_cluster_summary_gold.parquet` — one real row per issue-cluster
- `configs/bp4_customer_journey_analytics.yaml` — Gate 2 marker block appended/overwritten

## Prerequisites
BP4 Gate 1 must have been real-run at least once (`configs/bp4_customer_journey_analytics.yaml`
status must contain `gate1_confirmed`) — this notebook checks that live and raises a clear error
if it is missing, rather than silently re-deriving BP4's scope from nothing.

## If a structural check below fails
It raises `AssertionError` naming the failing check. A failing null-handling or row-accounting
check in particular must never be worked around — if real rows are unaccounted for after the
sentinel fill or the taxonomy join, that is a real data-integrity risk, not a check to loosen.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
BP4_CONFIG_PATH = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402

import polars as pl  # noqa: E402
import yaml  # noqa: E402

from features.bp4_journey_features import (  # noqa: E402
    BARRED_JOURNEY_COLUMNS,
    CLUSTER_KEY,
    build_issue_cluster_monthly,
    build_issue_cluster_summary,
    feature_lineage_table,
    live_null_counts,
    load_cfpb_with_journey_features,
)
from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"
GOLD_TAXONOMY_PATH = DATA_PROCESSED_DIR / "cfpb_common_taxonomy_gold.parquet"

# ============================================================
# SECTION 4: Load Gate 1's real policy.json live and confirm the config yaml prerequisite - never
# re-derive BP4's scope from nothing.
# ============================================================
gate1_policy_path = ARTIFACTS_DIR / "policy.json"
assert (
    gate1_policy_path.exists()
), f"{gate1_policy_path} does not exist - BP4 Gate 1 must be real-run before Gate 2 can proceed."
with open(gate1_policy_path, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)

assert BP4_CONFIG_PATH.exists(), f"{BP4_CONFIG_PATH} does not exist - run Gate 1 first."
with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    _front_matter_only = yaml.safe_load(f.read().split("# --- Gate", 1)[0])
gate1_status_confirmed = "gate1_confirmed" in str(_front_matter_only.get("status", ""))
assert gate1_status_confirmed, (
    f"configs/bp4_customer_journey_analytics.yaml status is "
    f"'{_front_matter_only.get('status')}' - expected it to contain 'gate1_confirmed'. Run Gate 1 "
    "for real before Gate 2."
)
print(f"[OK] Gate 1 prerequisite confirmed: status='{_front_matter_only.get('status')}'")
assert not any(c in CLUSTER_KEY for c in BARRED_JOURNEY_COLUMNS), (
    "A barred journey column leaked into CLUSTER_KEY - see BARRED_JOURNEY_COLUMNS in "
    "src/features/bp4_journey_features.py."
)

# ============================================================
# SECTION 5: Live null screen on journey-relevant columns - cross-checked against Gate 1's
# recorded null_counts_journey_columns (zero drift expected, never assumed).
# ============================================================
cfpb_lazy = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
total_rows = cfpb_lazy.select(pl.len()).collect().item()
live_null_report = live_null_counts(cfpb_lazy)
print("\n=== LIVE null counts, journey-relevant columns ===")
print(live_null_report)

gate1_null_counts = gate1_policy["live_checks"]["null_counts_journey_columns"]
null_count_drift = []
for row in live_null_report.to_dicts():
    col, live_count = row["column"], row["null_count"]
    if col in gate1_null_counts and gate1_null_counts[col] != live_count:
        null_count_drift.append((col, gate1_null_counts[col], live_count))
print(f"[OK] Live vs Gate 1 null-count drift: {null_count_drift or 'NONE'}")

# ============================================================
# SECTION 6: Live re-check of the Gate 2 compliance touchpoint (narrative-text / PII screen) -
# confirmed live, not assumed carried over from Gate 1.
# ============================================================
narrative_columns_found = [c for c in cfpb_lazy.collect_schema().names() if "narrative" in c.lower()]
print(f"[OK] Narrative-text columns present (live re-check): {narrative_columns_found or 'NONE'}")

# ============================================================
# SECTION 7: Build the real complaint-event journey (row-level) via the new shared module.
# ============================================================
journey_lazy = load_cfpb_with_journey_features(CFPB_PATH, GOLD_TAXONOMY_PATH)
journey_row_count = journey_lazy.select(pl.len()).collect().item()
journey_row_count_matches_raw = journey_row_count == total_rows
print(f"[OK] Real complaint-event journey row count (live): {journey_row_count:,}")
print(
    "[OK] Journey row count matches raw CFPB row count (no join fan-out/drop): "
    f"{journey_row_count_matches_raw}"
)

# ============================================================
# SECTION 8: Build the real issue-cluster journey (aggregate-level: monthly + summary tables).
# ============================================================
cluster_monthly_lazy = build_issue_cluster_monthly(journey_lazy)
cluster_summary_lazy = build_issue_cluster_summary(journey_lazy)

cluster_summary_df = cluster_summary_lazy.collect()
n_clusters_gate2 = cluster_summary_df.height
n_recurring_clusters_gate2 = int(cluster_summary_df["is_recurring_cluster"].sum())
rows_in_recurring_clusters_gate2 = int(
    cluster_summary_df.filter(pl.col("is_recurring_cluster"))["n_complaints_total"].sum()
)
print(
    f"[OK] Real issue-cluster summary (live): {n_clusters_gate2:,} clusters, "
    f"{n_recurring_clusters_gate2:,} recurring, {rows_in_recurring_clusters_gate2:,} rows in "
    "recurring clusters"
)

gate1_cluster_stats = gate1_policy["live_checks"]["issue_cluster_stats"]
cluster_count_matches_gate1 = n_clusters_gate2 == gate1_cluster_stats["n_clusters"]
print(
    f"[OK] Gate 2 cluster count matches Gate 1's recorded {gate1_cluster_stats['n_clusters']:,} "
    f"(sentinel fill produces the same grouping Polars' own null-grouping already gave Gate 1): "
    f"{cluster_count_matches_gate1}"
)

cluster_monthly_df = cluster_monthly_lazy.collect()
cluster_monthly_row_count = cluster_monthly_df.height
print(f"[OK] Real issue-cluster monthly rows (live): {cluster_monthly_row_count:,}")

# ============================================================
# SECTION 9: Write the feature-lineage table (Gate 2's own named exit-criterion artifact).
# ============================================================
lineage = feature_lineage_table()
print("\n=== FEATURE LINEAGE TABLE ===")
print(lineage)

lineage_path = ARTIFACTS_DIR / "gate2_feature_lineage.csv"
lineage.write_csv(lineage_path)
print(f"[SAVED] {lineage_path.relative_to(PROJECT_ROOT)}")

no_barred_column_in_lineage_as_feature = (
    lineage.filter(
        pl.col("source_column").is_in(BARRED_JOURNEY_COLUMNS)
        & (pl.col("engineered_feature") != "(none - barred from every BP4 journey-grouping key)")
    ).height
    == 0
)

# ============================================================
# SECTION 10: Write the three real Gold-layer Parquet outputs (WARP: Parquet, lazy sink where
# the frame is still lazy).
# ============================================================
journey_event_gold_path = DATA_PROCESSED_DIR / "cfpb_journey_event_gold.parquet"
journey_lazy.sink_parquet(journey_event_gold_path)
journey_event_gold_rows = pl.scan_parquet(journey_event_gold_path).select(pl.len()).collect().item()
print(f"[SAVED] {journey_event_gold_path.relative_to(PROJECT_ROOT)} ({journey_event_gold_rows:,} rows)")

cluster_monthly_gold_path = DATA_PROCESSED_DIR / "cfpb_issue_cluster_monthly_gold.parquet"
cluster_monthly_df.write_parquet(cluster_monthly_gold_path)
print(f"[SAVED] {cluster_monthly_gold_path.relative_to(PROJECT_ROOT)} ({cluster_monthly_df.height:,} rows)")

cluster_summary_gold_path = DATA_PROCESSED_DIR / "cfpb_issue_cluster_summary_gold.parquet"
cluster_summary_df.write_parquet(cluster_summary_gold_path)
print(f"[SAVED] {cluster_summary_gold_path.relative_to(PROJECT_ROOT)} ({cluster_summary_df.height:,} rows)")

# ============================================================
# SECTION 11: Write the Gate 2 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, fifth BP to do so).
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate2_marker = (
    "# --- Gate 2 (Data Verification & Feature/Taxonomy Engineering) results "
    "(appended, idempotent overwrite) ---"
)
gate2_block_lines = (
    [
        f"null_count_drift_vs_gate1: {'none' if not null_count_drift else 'DRIFT_DETECTED'}",
        f"narrative_text_column_found: {bool(narrative_columns_found)}",
        "journey_relevant_null_counts:",
    ]
    + [f"  {row['column']}: {row['null_count']}" for row in live_null_report.to_dicts()]
    + [
        f"journey_row_count: {journey_row_count}",
        f"journey_row_count_matches_raw: {journey_row_count_matches_raw}",
        f"n_clusters: {n_clusters_gate2}",
        f"n_recurring_clusters: {n_recurring_clusters_gate2}",
        f"cluster_count_matches_gate1: {cluster_count_matches_gate1}",
        f"cluster_monthly_row_count: {cluster_monthly_row_count}",
        f'feature_lineage_path: "{lineage_path.relative_to(PROJECT_ROOT).as_posix()}"',
        f"no_barred_column_used_as_feature: {no_barred_column_in_lineage_as_feature}",
        'journey_event_gold_path: "' + journey_event_gold_path.relative_to(PROJECT_ROOT).as_posix() + '"',
        f"journey_event_gold_rows_written: {journey_event_gold_rows}",
        'cluster_monthly_gold_path: "' + cluster_monthly_gold_path.relative_to(PROJECT_ROOT).as_posix() + '"',
        'cluster_summary_gold_path: "' + cluster_summary_gold_path.relative_to(PROJECT_ROOT).as_posix() + '"',
    ]
)
write_gate_block(BP4_CONFIG_PATH, gate2_marker, gate2_block_lines)
print(f"[SAVED] gate2 block written to {BP4_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "gate1_prerequisite_confirmed": gate1_status_confirmed,
    "no_barred_column_in_cluster_key": not any(c in CLUSTER_KEY for c in BARRED_JOURNEY_COLUMNS),
    "no_null_count_drift_vs_gate1": not null_count_drift,
    "no_narrative_text_column_found": len(narrative_columns_found) == 0,
    "journey_row_count_matches_raw_cfpb": journey_row_count_matches_raw,
    "cluster_count_matches_gate1_recorded_value": cluster_count_matches_gate1,
    "at_least_one_recurring_cluster": n_recurring_clusters_gate2 > 0,
    "cluster_monthly_rows_gte_cluster_count": cluster_monthly_row_count >= n_clusters_gate2,
    "no_barred_column_used_as_feature_in_lineage": no_barred_column_in_lineage_as_feature,
    "feature_lineage_csv_written": lineage_path.exists(),
    "journey_event_gold_row_count_matches_raw": journey_event_gold_rows == total_rows,
    "journey_event_gold_written": journey_event_gold_path.exists(),
    "cluster_monthly_gold_written": cluster_monthly_gold_path.exists(),
    "cluster_summary_gold_written": cluster_summary_gold_path.exists(),
    "cluster_summary_row_count_matches_live": cluster_summary_gold_path.exists()
    and n_clusters_gate2 == cluster_summary_df.height,
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP4 Gate 2 complete - real complaint-event journey built "
    f"({journey_event_gold_rows:,} rows, zero dropped), real issue-cluster journey built "
    f"({n_clusters_gate2:,} clusters, {n_recurring_clusters_gate2:,} recurring, matching Gate 1's "
    "recorded count exactly), every journey-relevant null explicitly sentinel-filled and traced, "
    "feature-lineage table written. Proceed to BP4 Gate 3 (Model/Classifier Benchmark & Champion "
    "Selection - for BP4, the Polars/DuckDB aggregation-pipeline benchmark per Master Plan "
    "Section 17.5) next."
)
